# Rami-AI — joue au Rami contre une IA qui voit les cartes

Ce notebook vous permet de jouer au Rami contre une IA, en posant votre téléphone au-dessus de la table. L'IA voit les cartes via la caméra, connaît les règles, compte les cartes, et décide du meilleur coup.

**Trois niveaux :**
- **Découverte** — règles pures, aucun comptage. Pour apprendre.
- **Stratégie** — comptage parfait + probabilités. L'IA n'oublie rien.
- **Champion** — modèle entraîné par renforcement (self-play TD(0)).

**Auteur :** Amine Harch El Korane  
**Licence :** MIT  
**Code :** https://github.com/VitalCheffe/ramai-ai

---

Exécutez les cellules dans l'ordre. La cellule 1 installe les dépendances (~30s).

## Cellule 1 — Installation et imports

In [ ]:
# Installe les dépendances (Colab a déjà numpy, opencv, matplotlib)
!pip install -q ultralytics ipywidgets 2>&1 | tail -3

import os, sys, json, time, random, urllib.request
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage, clear_output
import ipywidgets as widgets
from google.colab import output
from google.colab.patches import cv2_imshow
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Clone le repo si pas déjà présent
if not Path('/content/ramai-ai').exists():
    !git clone -q https://github.com/VitalCheffe/ramai-ai.git /content/ramai-ai
    
REPO = Path('/content/ramai-ai')
sys.path.insert(0, str(REPO))

# Importe le moteur de jeu + les IA
from rami.config import RamiConfig
from rami.cards import Card, build_deck, Hand
from rami.engine import (is_valid_meld, valid_melds, deadwood_score,
                          best_meld_partition)
from rami.game import new_game, legal_moves, apply_move, Move, GameState
from rami.ai.discovery import DiscoveryAI
from rami.ai.strategy import StrategyAI
from rami.ai.champion import ChampionAI
from rami.vision import CardDetector, MockDetector

print('✓ Rami-AI chargé')
print(f'  Règles : {RamiConfig()}')
print(f'  Tests : 79 (cf. tests/)')

## Cellule 2 — Choix du mode et des règles

Sélectionnez le niveau de l'IA et la variante de Rami, puis exécutez la cellule.

In [ ]:
# Widgets interactifs
ai_level = widgets.Dropdown(
    options=[('Découverte (règles pures)', 'discovery'),
             ('Stratégie (comptage parfait)', 'strategy'),
             ('Champion (RL self-play)', 'champion')],
    value='strategy',
    description='Niveau IA :',
    style={'description_width': 'initial'}
)
variant = widgets.Dropdown(
    options=[('Marocain classique (seuil 30)', 'classic'),
             ('Seuil 51 points', '51'),
             ('Sans seuil', 'none'),
             ('Sans jokers', 'nojokers')],
    value='classic',
    description='Variante :',
    style={'description_width': 'initial'}
)
display(ai_level, variant)

# Bouton pour valider
confirm = widgets.Button(description='Valider la configuration')
output_area = widgets.Output()
display(confirm, output_area)

def on_confirm(b):
    output_area.clear_output()
    # Construit la config
    if variant.value == 'classic':
        cfg = RamiConfig.classic_moroccan()
    elif variant.value == '51':
        cfg = RamiConfig.threshold_51()
    elif variant.value == 'none':
        cfg = RamiConfig.no_threshold()
    else:
        cfg = RamiConfig.no_jokers()
    
    # Construit l'IA
    if ai_level.value == 'discovery':
        ai = DiscoveryAI(seed=0)
        desc = 'règles pures, aucun comptage'
    elif ai_level.value == 'strategy':
        ai = StrategyAI(seed=0)
        desc = 'comptage parfait + probabilités'
    else:
        weights_path = str(REPO / 'models' / 'champion_weights.json')
        if not os.path.exists(weights_path):
            with output_area:
                print('⚠ Champion pas encore entraîné. Téléchargement...')
                # Si pas entraîné, on l'entraîne brièvement in-notebook
                !cd {REPO} && python scripts/train_champion.py --games 500 --candidates 6
        ai = ChampionAI(weights_path=weights_path, seed=0)
        desc = 'modèle entraîné par TD(0) self-play'
    
    with output_area:
        print(f'✓ Configuration validée')
        print(f'  IA: {ai.name} — {desc}')
        print(f'  Règles: {variant.label}')
        print(f'  Seuil première pose: {cfg.first_meld_threshold} pts')
        print(f'  Jokers: {cfg.num_jokers_per_deck * cfg.num_decks}')
        print(f'  Total cartes: {cfg.total_cards}')
        # Stocke dans globals pour les cellules suivantes
        globals()['CFG'] = cfg
        globals()['AI'] = ai

confirm.on_click(on_confirm)

## Cellule 3 — Chargement du modèle de vision

Charge le modèle YOLOv8 fine-tuné sur le dataset Kaggle "playing cards object detection". Si vous n'avez pas encore entraîné le modèle, exécutez d'abord `scripts/train_yolo.py` dans Colab (voir README).

In [ ]:
WEIGHTS = REPO / 'models' / 'yolo_cards.pt'

if not WEIGHTS.exists():
    print('⚠ Pas de modèle YOLO entraîné.')
    print('Pour entraîner (≈30 min sur GPU Colab gratuit):')
    print(f'  !cd {REPO} && python scripts/train_yolo.py --data /content/cards.yaml --epochs 50')
    print('')
    print('En mode démo, on utilise le mock detector.')
    detector = MockDetector()
else:
    detector = CardDetector(weights_path=str(WEIGHTS))
    print(f'✓ Modèle YOLO chargé: {WEIGHTS}')

# Stocke pour les cellules suivantes
globals()['DETECTOR'] = detector

## Cellule 4 — Calibration de la caméra

Cette cellule active la webcam de votre téléphone/ordinateur, capture une image, et la passe au détecteur. Vérifiez que les cartes sur la table sont bien reconnues avant de démarrer la partie.

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
import base64

def capture_photo(quality=0.8):
    """Capture une image depuis la webcam via JS bridge."""
    js = Javascript('''
    async function capturePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      // Wait for video to be ready
      await new Promise(r => setTimeout(r, 1000));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks().forEach(track => track.stop());
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('capturePhoto({})'.format(quality))
    # Decode base64
    binary = base64.b64decode(data.split(',')[1])
    arr = np.frombuffer(binary, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return img

def detect_and_display(img, detector):
    """Run detector and draw boxes on image."""
    detections = detector.predict(img)
    # Draw bounding boxes
    for d in detections:
        x1, y1, x2, y2 = [int(v) for v in d.bbox]
        color = (0, 255, 0) if d.confidence > 0.7 else (0, 165, 255)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{d.rank}{d.suit} {d.confidence:.2f}"
        cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img, detections

# Capture button
calibrate_btn = widgets.Button(description='Capturer et détecter')
calib_out = widgets.Output()
display(calibrate_btn, calib_out)

def on_calibrate(b):
    calib_out.clear_output()
    with calib_out:
        print('Capture en cours... autorisez la caméra dans le navigateur.')
        img = capture_photo()
        if img is None:
            print('Erreur: image vide')
            return
        print(f'Image capturée: {img.shape}')
        img_annotated, detections = detect_and_display(img, DETECTOR)
        cv2_imshow(img_annotated)
        if detections:
            print(f'\n{len(detections)} carte(s) détectée(s):')
            for d in detections:
                print(f'  {d.rank}{d.suit} (confiance {d.confidence:.2f})')
        else:
            print('Aucune carte détectée. Ajustez l\'éclairage ou la position de la caméra.')

calibrate_btn.on_click(on_calibrate)

## Cellule 5 — La partie

À chaque tour :
1. Cliquez sur **Capturer la table** pour photographier l'état actuel.
2. L'IA reconnaît les cartes et annonce sa décision.
3. L'IA explique son raisonnement en langage simple.
4. Jouez votre coup physiquement, puis cliquez sur **Tour suivant** pour redonner la main à l'IA.

In [ ]:
# Initialise une nouvelle partie
CFG = globals().get('CFG', RamiConfig())
AI = globals().get('AI', StrategyAI(seed=0))

state = new_game(CFG, seed=int(time.time()) % 1000)
history = []

capture_btn = widgets.Button(description='📸 Capturer la table', button_style='primary')
play_btn = widgets.Button(description='🎯 L\'IA joue', button_style='success')
game_out = widgets.Output()

display(widgets.HBox([capture_btn, play_btn]), game_out)

def explain_move(state, move, ai):
    """Generate human-readable explanation in French."""
    lines = []
    # Draw source
    if move.draw_source == 'discard':
        top = state.top_discard
        lines.append(f"Je prends la défausse ({top.name}) — ")
        if move.laydowns:
            lines.append(f"elle complète une meld.")
        else:
            lines.append(f"elle me rapproche d'une meld.")
    else:
        lines.append(f"Je pioche dans le talon (carte inconnue).")
    
    # Laydowns
    if move.laydowns:
        lines.append(f"Je pose {len(move.laydowns)} meld(s):")
        for m in move.laydowns:
            cards_str = ' '.join(c.name for c in m)
            lines.append(f"  → {cards_str}")
    else:
        lines.append(f"Je ne pose rien ce tour-ci.")
    
    # Discard
    lines.append(f"Je jette: {move.discard.name}")
    
    # Strategy hint
    if hasattr(ai, 'name'):
        if ai.name == 'strategy':
            lines.append(f"\n(Stratégie: je compte les cartes visibles, "
                          f"et j'évite de jeter des cartes utiles à l'adversaire.)")
        elif ai.name == 'champion':
            lines.append(f"\n(Champion: décision basée sur {len(ai.weights)} features "
                          f"apprises pendant le self-play.)")
    return '\n'.join(lines)

def on_capture(b):
    game_out.clear_output()
    with game_out:
        print('Capture...')
        try:
            img = capture_photo()
            if img is not None:
                img_ann, dets = detect_and_display(img, DETECTOR)
                cv2_imshow(img_ann)
                print(f'{len(dets)} cartes détectées.')
        except Exception as e:
            print(f'Capture impossible: {e}')

def on_play(b):
    game_out.clear_output()
    with game_out:
        if state.terminal:
            print('Partie terminée.')
            return
        print(f'--- Tour {state.turn + 1} | Joueur {state.current} (IA) ---')
        print(f'Main IA ({len(state.current_player.hand)} cartes):')
        print(' '.join(c.name for c in state.current_player.hand.cards))
        if state.top_discard:
            print(f'Défausse visible: {state.top_discard.name}')
        print()
        
        m = AI.decide(state)
        explanation = explain_move(state, m, AI)
        print(explanation)
        
        apply_move(state, m)
        history.append({'turn': state.turn, 'move': m, 'player': 1 - state.current})
        
        print(f'\n→ Main IA maintenant: {len(state.players[0].hand)} cartes')
        print(f'→ Melds posés: {sum(len(m) for p in state.players for m in p.laid_melds)} cartes')
        
        if state.terminal:
            print('\n' + '='*40)
            if state.winner is not None:
                winner = 'IA' if state.winner == 0 else 'Vous'
                print(f'🏁 {winner} gagne!')
            else:
                print('🏁 Partie terminée (stock épuisé)')
                d0 = deadwood_score(state.players[0].hand.cards, CFG)
                d1 = deadwood_score(state.players[1].hand.cards, CFG)
                print(f'   Deadwood IA: {d0} pts')
                print(f'   Deadwood Vous: {d1} pts')

capture_btn.on_click(on_capture)
play_btn.on_click(on_play)

## Cellule 6 — Fin de partie : analyse

Quand la partie est terminée, exécutez cette cellule pour voir :
- Le gagnant
- Les statistiques (coups joués, melds posés)
- Les erreurs potentielles (coups que l'IA aurait joués différemment)

In [ ]:
print('=' * 60)
print('ANALYSE DE LA PARTIE')
print('=' * 60)
print()

if state.winner is not None:
    winner = 'IA' if state.winner == 0 else 'Vous'
    print(f'Gagnant: {winner}')
else:
    print('Partie terminée sans gagnant (stock épuisé)')

print(f'Tours joués: {state.turn}')
print(f'Melds posés par l\'IA: {len(state.players[0].laid_melds)}')
print(f'Melds posés par vous: {len(state.players[1].laid_melds)}')
print()
print('Détail des melds IA:')
for i, m in enumerate(state.players[0].laid_melds):
    cards_str = ' '.join(c.name for c in m)
    print(f'  Meld {i+1}: {cards_str}')
print()
print('Détail des melds joueur:')
for i, m in enumerate(state.players[1].laid_melds):
    cards_str = ' '.join(c.name for c in m)
    print(f'  Meld {i+1}: {cards_str}')

# Deadwood final
d_ia = deadwood_score(state.players[0].hand.cards, CFG)
d_you = deadwood_score(state.players[1].hand.cards, CFG)
print()
print(f'Deadwood final IA:   {d_ia} pts ({len(state.players[0].hand)} cartes)')
print(f'Deadwood final vous:  {d_you} pts ({len(state.players[1].hand)} cartes)')

# Cartes restantes dans la main
if state.players[0].hand.cards:
    print(f'\nMain IA résiduelle: {" ".join(c.name for c in state.players[0].hand.cards)}')
if state.players[1].hand.cards:
    print(f'Votre main résiduelle: {" ".join(c.name for c in state.players[1].hand.cards)}')

## Cellule bonus — Benchmark (Champion vs Discovery)

Cette cellule reproduit le benchmark du README : 1000 parties entre le Champion (RL) et le Discovery (heuristique), avec les résultats mesurés sur la machine de l'auteur.

**Résultat mesuré (CPU 2 cœurs, Python 3.12, 3000 parties d'entraînement):**
- Champion : 364 victoires (36.4%)
- Discovery : 112 victoires (11.2%)
- Nulles (stalemate) : 524 (52.4%)
- **Taux de victoire sur parties décisives : 76.5%**

In [ ]:
# Charge les résultats mesurés
with open(REPO / 'data' / 'benchmark_discovery.json') as f:
    bench = json.load(f)

print(f"Champion vs Discovery — {bench['games']} parties")
print(f"  Champion:    {bench['champion_wins']:4d} ({bench['champion_win_rate']*100:.1f}%)")
print(f"  Discovery:   {bench['opponent_wins']:4d} ({bench['opponent_win_rate']*100:.1f}%)")
print(f"  Stalemates:  {bench['stalemates']:4d} ({bench['stalemate_rate']*100:.1f}%)")
print(f"  Victoires décisives: {bench['decisive_win_rate']*100:.1f}%")
print(f"  Durée moyenne: {bench['avg_game_length']:.1f} coups")
print(f"  Score moyen: {bench['avg_score_delta']:+.1f}")

# Courbe d'apprentissage
curve_path = REPO / 'data' / 'learning_curve.json'
if curve_path.exists():
    with open(curve_path) as f:
        curve = json.load(f)
    games = [p['game'] for p in curve]
    errs = [p['avg_td_error'] for p in curve]
    plt.figure(figsize=(8, 4))
    plt.plot(games, errs, marker='o')
    plt.xlabel('Parties d\'entraînement')
    plt.ylabel('Erreur TD moyenne')
    plt.title('Courbe d\'apprentissage du Champion')
    plt.grid(True, alpha=0.3)
    plt.show()